In [6]:
import altair as alt
import pandas as pd
import geopandas as gpd # Requires geopandas -- e.g.: conda install -c conda-forge geopandas
alt.data_transformers.enable('default', max_rows=None) # Inline data so static rendering can access it
alt.renderers.enable('png') # Static renderer: avoids frontend vega-embed/requirejs issues in VS Code notebooks
import ipywidgets as widgets
from IPython.display import clear_output, display
from pathlib import Path

In [7]:
data_path = Path('dpt2020.csv')
if not data_path.exists():
    data_path = Path('Names hints') / 'dpt2020.csv'
just_names = pd.read_csv(data_path, sep=';')
just_names = just_names[just_names['preusuel'] != '_PRENOMS_RARES']
just_names = just_names[just_names['dpt'] != 'XX']

years = sorted(just_names['annais'].astype(str).str.strip().astype(int).unique())

# Fixed color map for all first names (stable across years)
all_names = sorted(just_names['preusuel'].dropna().astype(str).unique())
base_colors = [
    '#4C78A8', '#F58518', '#E45756', '#72B7B2', '#54A24B',
    '#EECA3B', '#B279A2', '#FF9DA6', '#9D755D', '#BAB0AC',
    '#1F77B4', '#FF7F0E', '#2CA02C', '#D62728', '#9467BD',
    '#8C564B', '#E377C2', '#7F7F7F', '#BCBD22', '#17BECF',
    '#264653', '#2A9D8F', '#E9C46A', '#F4A261', '#E76F51',
    '#3A86FF', '#8338EC', '#FF006E', '#FB5607', '#06D6A0',
    '#118AB2', '#073B4C', '#8E9AAF', '#CBC0D3', '#EFD3D7'
]
color_range = [base_colors[i % len(base_colors)] for i in range(len(all_names))]

# Fixed x-axis scale for all years: global max + 5000
max_nombre_global = (
    just_names
    .groupby(['annais', 'preusuel'], as_index=False)['nombre']
    .sum()['nombre']
    .max()
)
x_max = int(max_nombre_global) + 5000

# Precompute yearly stats once for faster updates
yearly_name_totals = (
    just_names.assign(annais_clean=just_names['annais'].astype(str).str.strip().astype(int))
    .groupby(['annais_clean', 'preusuel'], as_index=False)['nombre']
    .sum()
)
annual_stats = (
    yearly_name_totals
    .groupby('annais_clean', as_index=False)
    .agg(
        births_this_year=('nombre', 'sum'),
        names_over_1000=('nombre', lambda series: int((series > 1000).sum()))
    )
    .set_index('annais_clean')
)

# Precompute per-year payload once: cards + chart
cards_by_year = {}
charts_by_year = {}

for annee in years:
    year_data = yearly_name_totals[yearly_name_totals['annais_clean'] == annee]
    top10_annee = (
        year_data
        .drop(columns=['annais_clean'])
        .sort_values('nombre', ascending=False)
        .head(10)
    )

    births_this_year = int(annual_stats.loc[annee, 'births_this_year'])
    names_over_1000 = int(annual_stats.loc[annee, 'names_over_1000'])
    top1_annee = top10_annee.head(1)

    top_card = f'''<div style="width:700px;height:80px;background:#ffd166;border-radius:0px;display:flex;justify-content:center;align-items:center;font-family:sans-serif;color:#1f2937;font-size:22px;font-weight:700;">
    Prénom de l'année : {top1_annee.iloc[0]['preusuel'] if len(top1_annee) else ''}
</div>'''

    def stat_html(label, value):
        return f'''<div style="width:340px;height:60px;background:#e5e7eb;border-radius:6px;display:flex;flex-direction:column;justify-content:center;align-items:center;font-family:sans-serif;color:#1f2937;">
    <div style="font-size:10px;color:#4b5563;line-height:1.05;">{label}</div>
    <div style="font-size:16px;font-weight:700;line-height:1.05;">{value}</div>
</div>'''

    stats_left = stat_html('Naissances cette année', f"{births_this_year:,}".replace(',', ' '))
    stats_right = stat_html('Prénoms > 1000 naissances', str(names_over_1000))
    cards_by_year[annee] = (top_card, stats_left, stats_right)

    name_order = top10_annee['preusuel'].tolist()
    bars = alt.Chart(top10_annee).mark_bar().encode(
        x=alt.X(
            'nombre:Q',
            title='Nombre de naissances',
            axis=alt.Axis(domain=False),
            scale=alt.Scale(domain=[0, x_max])
        ),
        y=alt.Y(
            'preusuel:N',
            sort=name_order,
            title='Prénom',
            axis=alt.Axis(domain=False)
        ),
        color=alt.Color(
            'preusuel:N',
            legend=None,
            scale=alt.Scale(domain=all_names, range=color_range)
        ),
        tooltip=['preusuel:N', 'nombre:Q']
    ) + alt.Chart(top10_annee).mark_text(
        align='left',
        baseline='middle',
        dx=5
    ).encode(
        x='nombre:Q',
        y=alt.Y(
            'preusuel:N',
            sort=name_order,
            axis=alt.Axis(domain=False)
        ),
        text=alt.Text('nombre:Q', format=',d')
    )

    charts_by_year[annee] = bars.properties(
        title=f'Top 10 des prénoms en {annee}',
        width=700,
        height=350
    )

In [11]:
annee_slider = widgets.IntSlider(
    value=1900,
    min=years[0],
    max=years[-1],
    step=1,
    description='Année',
    continuous_update=False,
    readout=True,
    readout_format='d',
    tooltip='Choisir l année affichée'
)

vitesse_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=2.0,
    step=0.1,
    description='Vitesse (s)',
    continuous_update=False,
    readout=True,
    readout_format='.1f',
    tooltip='Durée en secondes par année'
)

btn_play = widgets.ToggleButton(
    value=False,
    description='',
    icon='play',
    tooltip='Lancer le défilement des années',
    layout=widgets.Layout(width='44px')
)


play_engine = widgets.Play(
    value=annee_slider.value,
    min=annee_slider.min,
    max=annee_slider.max,
    step=1,
    interval=int(vitesse_slider.value * 1000),
    description=''
)

play_engine.layout = widgets.Layout(width='0px', height='0px', overflow='hidden')
widgets.jslink((play_engine, 'value'), (annee_slider, 'value'))

top_card_widget = widgets.HTML()
stats_left_widget = widgets.HTML()
stats_right_widget = widgets.HTML()
chart_out = widgets.Output()

In [14]:
def render_all():
    """Render all components for the current year (manual slider change)"""
    year = annee_slider.value
    
    was_playing = play_engine.playing
    if was_playing:
        play_engine.playing = False
    
    # 1. Display chart first
    with chart_out:
        clear_output(wait=True)
        display(charts_by_year[year])
    
    # 2. Stats are shown (year already updated by jslink from play_engine)
    top_card, stats_left, stats_right = cards_by_year[year]
    top_card_widget.value = top_card
    stats_left_widget.value = stats_left
    stats_right_widget.value = stats_right
    
    if was_playing:
        play_engine.playing = True


def refresh_chart(_change=None):
    """Called when slider changes manually"""
    render_all()


def update_play_interval(_change):
    play_engine.interval = int(vitesse_slider.value * 1000)


def on_play_step(change):
    """Handle each step of the Play - strict sequential order"""
    if change['name'] != 'value':
        return
    
    new_year = int(change['new'])
    
    # Pause play during rendering
    play_engine.playing = False
    
    # 1. Display chart for the new year
    with chart_out:
        clear_output(wait=True)
        display(charts_by_year[new_year])
    
    # 2. Update slider to show the year (jslink will handle sync)
    annee_slider.value = new_year
    
    # 3. Display stats for the new year
    top_card, stats_left, stats_right = cards_by_year[new_year]
    top_card_widget.value = top_card
    stats_left_widget.value = stats_left
    stats_right_widget.value = stats_right
    
    # Resume play for next iteration
    play_engine.playing = True


def on_play_toggle(change):
    if change['name'] != 'value':
        return

    if change['new'] and annee_slider.value >= annee_slider.max:
        annee_slider.value = annee_slider.min
        play_engine.value = annee_slider.value

    # Keep this line exactly as requested.
    play_engine.playing = bool(change['new'])
    btn_play.icon = 'pause' if change['new'] else 'play'
    btn_play.tooltip = (
        'Arrêter le défilement des années'
        if change['new']
        else 'Lancer le défilement des années'
    )


annee_slider.observe(refresh_chart, names='value')
play_engine.observe(on_play_step, names='value')
vitesse_slider.observe(update_play_interval, names='value')
btn_play.observe(on_play_toggle, names='value')

# Center cards and chart block on the same visual width.
stats_box = widgets.HBox(
    [stats_left_widget, stats_right_widget],
    layout=widgets.Layout(width='700px', justify_content='space-between', gap='16px')
)

# Add a small white margin around the chart.
chart_box = widgets.Box(
    [chart_out],
    layout=widgets.Layout(width='724px', padding='12px', margin='0px')
)

render_all()
display(widgets.VBox([
    top_card_widget,
    stats_box,
    chart_box,
    widgets.HBox([btn_play, annee_slider, vitesse_slider]),
    play_engine
], layout=widgets.Layout(align_items='center')))